# Partial reconfiguration of a RISC-V accelerator slot — live on a PYNQ-Z2

**What is in the FPGA right now:** a single-cycle RV32I core (static, never changes) with a memory-mapped *custom function unit* slot at `0xF000_0000`. The slot is a **reconfigurable partition**: we can replace what is inside it while the rest of the chip keeps its configuration.

**The program never changes.** It writes `op_a = 6`, `op_b = 7` to the slot, reads the result back and banks it in data memory word 4. What that number *is* depends only on what is currently sitting in the slot.

| slot contents | partial bitstream | word 4 | word 5 |
|---|---|---|---|
| 32×32 multiplier | `config1_pblock_cfu_partial.bit` (111 KB) | 6 × 7 = **42** | high word = 0 |
| bitwise unit | `config2_pblock_cfu_partial.bit` (111 KB) | popcount(6) = **2** | 6 & 7 = 6 |

In [ ]:
import time, statistics
from pynq import Overlay, Bitstream
import pynq

ol = Overlay('config1.bit')                       # static design + multiplier in the slot
ctl, dbg = ol.axi_gpio_0.channel1, ol.axi_gpio_0.channel2
RST = 1 << 10
PART = {'multiplier': 'config1_pblock_cfu_partial.bit', 'bitwise': 'config2_pblock_cfu_partial.bit'}

def run():
    """Reset the core, let the program run, read back words 4 and 5."""
    ctl.write(RST, 0x7FF); ctl.write(0, 0x7FF); time.sleep(0.001)
    out = []
    for a in (4, 5):
        ctl.write(a, 0x3FF); out.append(dbg.read())
    return out

def swap(name):
    """Hold the core in reset, load a partial bitstream into the slot, return seconds taken."""
    ctl.write(RST, 0x7FF)
    t = time.perf_counter(); Bitstream(PART[name], partial=True).download()
    return time.perf_counter() - t

print(f'core clock {pynq.ps.Clocks.fclk0_mhz} MHz, static bitstream loaded, multiplier in the slot')

## 1. Run the program with the multiplier in the slot

In [ ]:
w4, w5 = run()
print(f'word 4 = {w4}   (6 x 7)')
print(f'word 5 = {w5}   (high word of the product)')

## 2. Swap the slot to the bitwise unit — nothing else in the chip changes

The core is held in reset while ~111 KB of configuration frames are written through the PCAP port; the static region is untouched.

In [ ]:
dt = swap('bitwise')
w4, w5 = run()
print(f'swap took {dt*1e3:.1f} ms')
print(f'word 4 = {w4}   (popcount of 6 = 0b110)')
print(f'word 5 = {w5}   (6 & 7)')
print('\nsame program, same static logic, same clock -- different accelerator, different answer')

## 3. And back

In [ ]:
dt = swap('multiplier')
w4, w5 = run()
print(f'swap took {dt*1e3:.1f} ms   word 4 = {w4}   word 5 = {w5}')

## 4. How fast, and compared with what?

Ten swaps, then one full-device reload for comparison. The reconfiguration time has three honest values: what the PYNQ Python API takes (converts the file and hands it to the kernel each time), what the kernel driver + DMA take on an already-converted file, and the raw PCAP bandwidth limit.

In [ ]:
times = []
cur = 'multiplier'
for i in range(10):
    cur = 'bitwise' if cur == 'multiplier' else 'multiplier'
    dt = swap(cur); w4, _ = run()
    ok = (w4 == 2) if cur == 'bitwise' else (w4 == 42)
    times.append(dt); print(f'  -> {cur:10s} {dt*1e3:6.1f} ms   word 4 = {w4:2d}  {"ok" if ok else "MISMATCH"}')

t = time.perf_counter(); Bitstream('config1.bit').download(); t_full = time.perf_counter() - t

import os
def raw(bit, partial):
    binf = os.path.basename(bit).replace('.bit', '.bin')
    ctl.write(RST, 0x7FF); open('/sys/class/fpga_manager/fpga0/flags', 'w').write('1' if partial else '0')
    t = time.perf_counter(); open('/sys/class/fpga_manager/fpga0/firmware', 'w').write(binf); return time.perf_counter() - t
r_part = statistics.median([raw(PART['bitwise'], True), raw(PART['multiplier'], True)] * 1)
r_full = raw('config1.bit', False)
sz_p, sz_f = os.path.getsize(PART['bitwise']), os.path.getsize('config1.bit')

print(f'\n{"":18s}{"partial":>12s}{"full reload":>14s}{"ratio":>8s}')
print(f'{"size":18s}{sz_p/1e3:9.0f} KB{sz_f/1e6:11.2f} MB{sz_f/sz_p:8.1f}x')
print(f'{"PYNQ API":18s}{statistics.median(times)*1e3:9.1f} ms{t_full*1e3:11.0f} ms{t_full/statistics.median(times):8.1f}x')
print(f'{"kernel driver+DMA":18s}{r_part*1e3:9.2f} ms{r_full*1e3:11.1f} ms{r_full/r_part:8.1f}x')
print(f'{"PCAP limit":18s}{sz_p/400e6*1e3:9.2f} ms{sz_f/400e6*1e3:11.1f} ms{sz_f/sz_p:8.1f}x')

## 5. Picture for the slide

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
labels = ['PYNQ API', 'kernel driver + DMA', 'PCAP limit']
part = [statistics.median(times)*1e3, r_part*1e3, sz_p/400e6*1e3]
full = [t_full*1e3, r_full*1e3, sz_f/400e6*1e3]
x = np.arange(3); fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.bar(x-0.18, full, 0.36, label=f'full reload ({sz_f/1e6:.1f} MB)', color='#9aa5b1')
ax.bar(x+0.18, part, 0.36, label=f'partial swap ({sz_p/1e3:.0f} KB)', color='#2b6cb0')
for i in range(3):
    ax.text(x[i]+0.18, part[i]*1.15, f'{part[i]:.3g} ms', ha='center', fontsize=9)
    ax.text(x[i]-0.18, full[i]*1.15, f'{full[i]:.3g} ms', ha='center', fontsize=9)
ax.set_yscale('log'); ax.set_ylim(0.1, 3000); ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('ms (log)')
ax.set_title(f'Accelerator swap vs full reload (PYNQ-Z2, {pynq.ps.Clocks.fclk0_mhz:.0f} MHz core)', fontsize=11)
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False); fig.tight_layout(); fig.savefig('pr_demo.png', dpi=200); plt.show()

In [ ]:
# Live website: serves demo/ from the board. Open http://<board-ip>:8000 in a browser on the same network.
exec(open('demo/board_server.py').read())
srv = start(8000)          # srv.shutdown() to stop